In [23]:
import pandas as pd
from pathlib import Path

df = pd.read_parquet(Path("../data/processed/player_weeks_clean.parquet"))
df = df.sort_values(["player_id", "season", "week"]).reset_index(drop=True)
df.shape

(38571, 161)

In [24]:
import numpy as np                                                    # for np.where

NEXT_SEASON, NEXT_WEEK = 2026, 12                                     # the week we're predicting (change each week)

sched = pd.read_parquet("../data/raw/schedules_2020_2026.parquet")   # all games, including unplayed ones
games = sched[(sched["season"] == NEXT_SEASON) &                      # this season...
              (sched["week"] == NEXT_WEEK) &                          # ...the upcoming week...
              (sched["game_type"] == "REG")]                          # ...regular season only

# each game -> two rows, one from each team's point of view
home_side = games.assign(team=games["home_team"], opponent_team=games["away_team"],   # home team's view
                         is_home=1, rest_days=games["home_rest"])
away_side = games.assign(team=games["away_team"], opponent_team=games["home_team"],   # away team's view
                         is_home=0, rest_days=games["away_rest"])
sides = pd.concat([home_side, away_side])                             # one row per team playing this week
sides["implied_total"] = np.where(sides["is_home"] == 1,              # same formula as notebook 02:
                                  (sides["total_line"] + sides["spread_line"]) / 2,   #   home expected points
                                  (sides["total_line"] - sides["spread_line"]) / 2)   #   away expected points

# who to predict: anyone who played this season in the last 3 weeks, on his most recent team
recent = df[(df["season"] == NEXT_SEASON) & (df["week"] >= NEXT_WEEK - 3)]  # games from the last 3 weeks
latest = (recent.sort_values("week").groupby("player_id").tail(1)           # each player's most recent game
          [["player_id", "player_display_name", "position", "team"]])       # keep only who he is + his team

upcoming = latest.merge(                                              # attach his team's upcoming game
    sides[["game_id", "team", "opponent_team", "home_team", "away_team",
           "total_line", "spread_line", "home_rest", "away_rest",
           "is_home", "rest_days", "implied_total"]],
    on="team", how="inner")                                           # inner: drops players whose team has a bye
upcoming["season"], upcoming["week"] = NEXT_SEASON, NEXT_WEEK         # label the new rows with season + week
upcoming["is_upcoming"] = 1                                           # mark them as "not played yet"

df["is_upcoming"] = 0                                                 # every existing row has been played
df = (pd.concat([df, upcoming], ignore_index=True)                    # stack the new rows under the old ones
        .sort_values(["player_id", "season", "week"])                 # re-sort so shift/rolling see the right order
        .reset_index(drop=True))                                      # tidy row numbers

print(f"{len(upcoming):,} upcoming player-games added")               # expect several hundred
upcoming["position"].value_counts()                                   # by position

457 upcoming player-games added


position
WR    188
RB    115
TE    103
QB     51
Name: count, dtype: int64

In [25]:
cols = ["fantasy_points_ppr", "targets", "receptions", "carries", "offense_pct",
        "attempts", "passing_yards", "passing_tds", "passing_interceptions",
        "passing_epa", "rushing_yards"]

g = df.groupby(["player_id", "season"])

for c in cols:
    df[f"{c}_last3"] = g[c].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())

df[[c + "_last3" for c in cols]].describe().round(2)

,fantasy_points_ppr_last3,targets_last3,receptions_last3,carries_last3,offense_pct_last3,attempts_last3,passing_yards_last3,passing_tds_last3,passing_interceptions_last3,passing_epa_last3,rushing_yards_last3
count,34828.00,34828.00,34828.00,34828.00,34794.00,34828.00,34828.00,34828.00,34828.00,4221.00,34828.00
mean,7.97,2.99,2.03,2.49,0.52,3.10,22.04,0.14,0.07,1.03,10.83
std,6.74,2.84,1.98,4.68,0.30,9.62,69.32,0.49,0.26,6.47,21.77
min,-2.86,0.00,0.00,0.00,0.00,0.00,-3.00,0.00,0.00,-32.77,-14.00
25%,2.40,0.67,0.33,0.00,0.27,0.00,0.00,0.00,0.00,-2.63,0.00
50%,6.40,2.33,1.50,0.00,0.54,0.00,0.00,0.00,0.00,0.55,0.00
75%,12.15,4.67,3.00,2.67,0.78,0.00,0.00,0.00,0.00,4.87,10.33
max,44.50,21.00,14.00,32.00,1.00,59.00,474.67,5.00,4.00,33.19,173.00


In [26]:
import numpy as np                                        # for np.nan (blank value)

g = df.groupby(["player_id", "season"])                   # one group per player per season

# total EPA and total attempts over the previous 3 games (shift(1) = earlier games only, no leakage)
epa_sum = g["passing_epa"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).sum())  # sum of EPA, last 3 games
att_sum = g["attempts"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).sum())     # sum of attempts, last 3 games

df["epa_per_att_last3"] = epa_sum / att_sum               # efficiency: EPA per pass over those games
df.loc[att_sum < 20, "epa_per_att_last3"] = np.nan        # too few passes to trust -> blank

df.loc[df["position"] == "QB", "epa_per_att_last3"].describe().round(3)  # check QBs only; typical range is about -0.3 to +0.4

count    3342.000
mean        0.036
std         0.237
min        -1.655
25%        -0.104
50%         0.046
75%         0.195
max         0.851
Name: epa_per_att_last3, dtype: float64

In [27]:
import numpy as np                                            # for np.inf / np.nan

g = df.groupby(["player_id", "season"])                       # one group per player per season

# --- history ---
df["fantasy_points_ppr_last1"] = g["fantasy_points_ppr"].shift(1)   # last game's points (Baseline 1)
df["fantasy_points_ppr_season"] = g["fantasy_points_ppr"].transform(
    lambda s: s.shift(1).expanding().mean())                 # average of ALL earlier games this season

# --- efficiency: sum over last 3 games, then divide (same idea as EPA per attempt) ---
def last3_sum(col):                                           # helper: total of a stat over the previous 3 games
    return g[col].transform(lambda s: s.shift(1).rolling(3, min_periods=1).sum())  # shift(1) = earlier games only

rec, tgt = last3_sum("receptions"), last3_sum("targets")     # catches and targets, last 3 games
rec_yds = last3_sum("receiving_yards")                        # receiving yards, last 3 games
car, rush_yds = last3_sum("carries"), last3_sum("rushing_yards")  # carries and rushing yards, last 3 games

df["catch_rate_last3"] = rec / tgt                            # share of targets caught
df["yards_per_catch_last3"] = rec_yds / rec                   # receiving yards per catch
df["yards_per_carry_last3"] = rush_yds / car                  # rushing yards per carry

eff_cols = ["catch_rate_last3", "yards_per_catch_last3", "yards_per_carry_last3"]  # the 3 new ratio columns
df[eff_cols] = df[eff_cols].replace([np.inf, -np.inf], np.nan)  # dividing by 0 gives infinity -> make it blank

df[["fantasy_points_ppr_last1", "fantasy_points_ppr_season"] + eff_cols].describe().round(2)  # sanity check

,fantasy_points_ppr_last1,fantasy_points_ppr_season,catch_rate_last3,yards_per_catch_last3,yards_per_carry_last3
count,34828.00,34828.00,28407.00,27412.00,16275.00
mean,8.03,7.87,0.69,10.50,4.35
std,8.08,6.31,0.23,5.86,3.98
min,-4.44,-2.86,0.00,-10.00,-13.00
25%,1.60,2.70,0.56,7.00,2.67
50%,5.60,6.55,0.70,9.88,4.00
75%,12.46,12.00,0.85,13.17,5.43
max,57.90,44.50,1.00,88.00,71.00


In [28]:
show = ["player_display_name", "season", "week", "fantasy_points_ppr",   # who, when, actual points
        "fantasy_points_ppr_last1", "fantasy_points_ppr_last3",          # last game; last-3 average
        "fantasy_points_ppr_season"]
df[df["player_display_name"] == "Justin Jefferson"][show].head(6)

,player_display_name,season,week,fantasy_points_ppr,fantasy_points_ppr_last1,fantasy_points_ppr_last3,fantasy_points_ppr_season
22049,Justin Jefferson,2020,1,4.6,NaN,NaN,NaN
22050,Justin Jefferson,2020,2,7.4,4.6,4.600000,4.600000
22051,Justin Jefferson,2020,3,30.5,7.4,6.000000,6.000000
22052,Justin Jefferson,2020,4,14.3,30.5,14.166667,14.166667
22053,Justin Jefferson,2020,5,5.3,14.3,17.400000,14.200000
22054,Justin Jefferson,2020,6,39.6,5.3,16.700000,12.420000


In [29]:
# carry share: player's carries / team's carries that week
df["team_carries"] = df.groupby(["team", "season", "week"])["carries"].transform("sum")
df["carry_share"] = df["carries"] / df["team_carries"]

g = df.groupby(["player_id", "season"])      # rebuild the grouping so it sees the new column

share_cols = ["target_share", "air_yards_share", "wopr", "carry_share"]
for c in share_cols:
    df[f"{c}_last3"] = g[c].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())

df[[c + "_last3" for c in share_cols]].describe().round(2)

,target_share_last3,air_yards_share_last3,wopr_last3,carry_share_last3
count,34828.00,34828.00,34828.00,34828.00
mean,0.09,0.09,0.20,0.09
std,0.09,0.13,0.21,0.17
min,0.00,-0.39,-0.03,0.00
25%,0.02,0.00,0.04,0.00
50%,0.07,0.03,0.14,0.00
75%,0.15,0.15,0.32,0.10
max,0.59,0.97,1.52,0.98


In [30]:
# 1. total PPR points each defense allowed to each position, each week
allowed = (df.groupby(["opponent_team", "season", "week", "position"])["fantasy_points_ppr"]
             .sum().reset_index(name="ppr_allowed"))

# 2. that defense's average over all EARLIER weeks this season
allowed = allowed.sort_values(["opponent_team", "position", "season", "week"])
allowed["opp_ppr_allowed_avg"] = (allowed.groupby(["opponent_team", "position", "season"])["ppr_allowed"]
                                  .transform(lambda s: s.shift(1).expanding().mean()))

# 3. attach it to each player's row
df = df.merge(allowed[["opponent_team", "season", "week", "position", "opp_ppr_allowed_avg"]],
              on=["opponent_team", "season", "week", "position"], how="left", validate="many_to_one")

df.groupby("position")["opp_ppr_allowed_avg"].describe().round(1)

,count,mean,std,min,25%,50%,75%,max
position,,,,,,,,
QB,4075.0,16.9,3.9,1.0,14.6,16.8,19.1,38.8
RB,9566.0,22.3,5.0,4.3,19.1,22.1,25.0,52.8
TE,7681.0,12.3,4.1,0.0,9.6,12.2,14.7,35.2
WR,15283.0,34.0,6.7,6.1,29.8,33.8,37.6,79.4


In [31]:
# 1. total pass attempts per team per game (sum over that team's players)
team_pass = df.groupby(["team", "season", "week"])["attempts"].sum().reset_index(name="team_att")

# 2. each team's average over EARLIER games this season
team_pass = team_pass.sort_values(["team", "season", "week"])              # time order within each team
team_pass["team_att_avg"] = team_pass.groupby(["team", "season"])["team_att"].transform(
    lambda s: s.shift(1).expanding().mean())                              # shift = exclude this game

# 3. attach to each player's row by team and week
df = df.merge(team_pass[["team", "season", "week", "team_att_avg"]],
              on=["team", "season", "week"], how="left", validate="many_to_one")  # one team row per match

df["team_att_avg"].describe().round(1)                                    # check: mostly 30-38 per game

count    36605.0
mean        33.7
std          4.8
min         14.0
25%         30.5
50%         33.8
75%         36.7
max         59.0
Name: team_att_avg, dtype: float64

In [32]:
import numpy as np                                                        # for np.select (if / elif / else on columns)

no_history = df["fantasy_points_ppr_last1"].isna()                        # player's first game this season
team_first_week = df.groupby(["team", "season"])["week"].transform("min") # week of the team's first game this season
late_start = df["week"] > team_first_week                                 # player's first game came after his team's first game
played_before = df.groupby("player_id")["season"].transform("min") < df["season"]  # has rows in an earlier season

df["history_flag"] = np.select(                                           # pick the first condition that's True
    [~no_history,                                                         # has history          -> normal row
     no_history & ~late_start,                                            # first game = team's opener
     no_history & late_start & played_before,                             # late start, known player (injury return, suspension, backup's first touch)
     no_history & late_start & ~played_before],                           # late start, never seen before (rookie / new player)
    ["ok", "season_opener", "limitation_returning", "limitation_new"],    # the label for each case above
    default="ok")                                                         # fallback (not expected to be used)

print(df.groupby(["season", "history_flag"]).size().unstack(fill_value=0))  # count of each label per season

history_flag  limitation_new  limitation_returning    ok  season_opener
season                                                                 
2020                     252                     0  4941            350
2021                     112                   179  5233            342
2022                      84                   183  5200            341
2023                      66                   172  5224            339
2024                      77                   173  5275            339
2025                      69                   186  5427            355
2026                      59                   165  3528            357


In [33]:
#identifiers for each row (player-week)
id_cols = ["player_id", "player_display_name", "position", "team", "opponent_team", "season", "week", "history_flag", "is_upcoming"]

feature_cols = [
    # history
    "fantasy_points_ppr_last1", "fantasy_points_ppr_last3", "fantasy_points_ppr_season",
    # usage (previous 3 games)
    "targets_last3", "receptions_last3", "carries_last3", "offense_pct_last3",
    # role (previous 3 games)
    "target_share_last3", "air_yards_share_last3", "wopr_last3", "carry_share_last3",
    # passing (previous 3 games)
    "attempts_last3", "passing_yards_last3", "passing_tds_last3",
    "passing_interceptions_last3", "epa_per_att_last3", "rushing_yards_last3",
    # efficiency (previous 3 games)
    "catch_rate_last3", "yards_per_catch_last3", "yards_per_carry_last3",
    # team and opponent (earlier games this season)
    "team_att_avg", "opp_ppr_allowed_avg",
    # game context (known before kickoff, no shift needed)
    "implied_total", "total_line", "is_home", "rest_days",
]

target = "fantasy_points_ppr"

features = df[id_cols + feature_cols + [target]]
features.to_parquet("../data/processed/features.parquet", index=False)

print(features.shape)
features[feature_cols].isna().mean().round(3)     # share of blanks in each feature

(39028, 36)


fantasy_points_ppr_last1       0.108
fantasy_points_ppr_last3       0.108
fantasy_points_ppr_season      0.108
targets_last3                  0.108
receptions_last3               0.108
carries_last3                  0.108
offense_pct_last3              0.108
target_share_last3             0.108
air_yards_share_last3          0.108
wopr_last3                     0.108
carry_share_last3              0.108
attempts_last3                 0.108
passing_yards_last3            0.108
passing_tds_last3              0.108
passing_interceptions_last3    0.108
epa_per_att_last3              0.914
rushing_yards_last3            0.108
catch_rate_last3               0.272
yards_per_catch_last3          0.298
yards_per_carry_last3          0.583
team_att_avg                   0.062
opp_ppr_allowed_avg            0.062
implied_total                  0.000
total_line                     0.000
is_home                        0.000
rest_days                      0.000
dtype: float64